# CausalSelfAttention

How the model decides which earlier tokens are relevant and gathers information from them. Three steps: create Q/K/V, compute scores, gather information.

## Creating Q, K, V

Each token is a 512-dimensional vector. Three separate weight matrices project it into three new vectors:

- **Q (Query)** — "what am I looking for?" — 512 numbers (8 heads × 64 dims)
- **K (Key)** — "what do I advertise to others?" — 256 numbers (4 heads × 64 dims)
- **V (Value)** — "what I contribute when attended to" — 256 numbers (4 heads × 64 dims)

K and V are smaller than Q because of GQA (Grouped Query Attention). In the original transformer all three are the same size — 8 heads each, 64 dims each, all 512. Every query head gets its own dedicated key and value head.

GQA uses fewer KV heads and shares them across query heads. 8 query heads, 4 KV heads — each pair of query heads shares one key and one value head. Like 8 people asking questions but only 4 reference books in the room. They still ask different questions, they just consult the same sources.

The parameter savings matter in this competition:
- Full K matrix: 512 × 512 = 262,144 params
- GQA K matrix (4 heads): 512 × 256 = 131,072 params
- Same savings for V

Queries stay full size because you want each head asking its own distinct question. The answers can be shared.

```python
self.c_q = CastedLinear(dim, dim, bias=False)       # 512 → 512
self.c_k = CastedLinear(dim, kv_dim, bias=False)    # 512 → 256
self.c_v = CastedLinear(dim, kv_dim, bias=False)    # 512 → 256
```

## Reshaping Into Heads

After the matrix multiply, split the flat vector into separate heads:

```python
q = self.c_q(x).reshape(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)
```

1. `self.c_q(x)` — matrix multiply: (8, 1024, 512) → (8, 1024, 512)
2. `.reshape(bsz, seqlen, 8, 64)` — split 512 into 8 heads of 64
3. `.transpose(1, 2)` — put heads before sequence: (8, 8, 1024, 64)

No numbers change. Just rearranging the layout so PyTorch can process each head in parallel.

K and V reshape the same way but with 4 heads: (8, 4, 1024, 64).

## Prep: Normalize, RoPE, Query Gain

Before computing scores, Q and K get normalized (same pattern as everywhere — keep numbers in a stable range) and then RoPE is applied.

**RoPE** injects position information by rotating Q and K vectors. After RoPE, nearby tokens produce higher dot products, distant tokens produce lower ones. The model gets a built-in sense of distance. Mechanically: the 64-dim head vector splits into 32 pairs, each pair rotated by a position-dependent angle at different frequencies. Low-indexed pairs rotate fast (short-range position), high-indexed pairs rotate slowly (long-range position).

```python
q = F.rms_norm(q, (q.size(-1),))
k = F.rms_norm(k, (k.size(-1),))
cos, sin = self.rotary(seqlen, x.device, q.dtype)
q = apply_rotary_emb(q, cos, sin)
k = apply_rotary_emb(k, cos, sin)
```

**Query gain** scales each head's queries by a learned number (starts at 1.5). Heads can learn to turn this up for sharper attention or down for softer attention.

```python
q = q * self.q_gain[None, :, None, None]
```

## The Attention Computation

One PyTorch function does the entire `softmax(QK^T / √d_k) · V` formula:

```python
y = F.scaled_dot_product_attention(
    q, k, v,
    attn_mask=None,
    is_causal=True,
    enable_gqa=True,
)
```

What happens inside:
1. Each query (64 numbers) gets dot-producted against all keys (64 numbers each)
2. Divide by √64 to keep scores stable
3. Mask out future positions (`is_causal=True`) so tokens can't look ahead
4. Softmax so weights sum to 1
5. Multiply weights by values and sum up

`enable_gqa=True` handles the 8 query heads / 4 KV heads sharing.

Output is (8, 8, 1024, 64) — same shape as Q. Each head at each position now has 64 numbers representing what it found by attending to earlier tokens.

Concretely: if token 5 has weights 40% on token 2, 30% on token 3, 20% on token 1, 10% on token 0:
```
output[5] = 0.4 × value[2] + 0.3 × value[3] + 0.2 × value[1] + 0.1 × value[0]
```

## Recombining Heads

```python
y = y.transpose(1, 2).contiguous().reshape(bsz, seqlen, dim)
return self.proj(y)
```

Undo the earlier reshape: (8, 8, 1024, 64) → transpose → (8, 1024, 8, 64) → reshape → (8, 1024, 512). The 8 heads of 64 concatenate back into 512. Then `self.proj` is a final 512 → 512 matrix multiply that mixes the heads' outputs together.

This return value goes back to `Block.forward` and gets added to `x` via the residual connection.

## What's Built-in vs Custom

**Built-in PyTorch:** `F.rms_norm`, `reshape`, `transpose`, `F.scaled_dot_product_attention`, basic `*` multiplication.

**Custom in this repo:** `CastedLinear` (thin wrapper around `nn.Linear` that casts to bf16 for compute), `Rotary` and `apply_rotary_emb` (RoPE isn't a single PyTorch function).

Everything compiles down to the same basic operations — matrix multiplies, element-wise math, reshapes. The line between built-in and custom is just what PyTorch chose to ship.

## Full Code

```python
def forward(self, x: Tensor) -> Tensor:
    bsz, seqlen, dim = x.shape

    # Create Q, K, V and split into heads
    q = self.c_q(x).reshape(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)
    k = self.c_k(x).reshape(bsz, seqlen, self.num_kv_heads, self.head_dim).transpose(1, 2)
    v = self.c_v(x).reshape(bsz, seqlen, self.num_kv_heads, self.head_dim).transpose(1, 2)

    # Normalize, apply RoPE, scale queries
    q = F.rms_norm(q, (q.size(-1),))
    k = F.rms_norm(k, (k.size(-1),))
    cos, sin = self.rotary(seqlen, x.device, q.dtype)
    q = apply_rotary_emb(q, cos, sin)
    k = apply_rotary_emb(k, cos, sin)
    q = q * self.q_gain[None, :, None, None]

    # softmax(QK^T / sqrt(64)) * V, with causal mask and GQA
    y = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=None,
        is_causal=True,
        enable_gqa=(self.num_kv_heads != self.num_heads),
    )

    # Concatenate heads back to 512, project
    y = y.transpose(1, 2).contiguous().reshape(bsz, seqlen, dim)
    return self.proj(y)
```

Three matrix multiplies to create Q/K/V, prep work (normalize, RoPE, scale), one attention call, recombine and project back.

## Shape Reference

batch=8, seq=1024, dim=512, 8 query heads, 4 KV heads, head_dim=64:

| Variable | Shape | What it is |
|---|---|---|
| `x` (input) | (8, 1024, 512) | Token representations from Block |
| `self.c_q(x)` | (8, 1024, 512) | Raw Q before reshape |
| `self.c_k(x)` | (8, 1024, 256) | Raw K (GQA: half size) |
| `q` after reshape | (8, 8, 1024, 64) | 8 query heads, 64 dims each |
| `k` after reshape | (8, 4, 1024, 64) | 4 KV heads, 64 dims each |
| `v` after reshape | (8, 4, 1024, 64) | Same as K |
| `y` (attention out) | (8, 8, 1024, 64) | Per-head gathered information |
| `y` after reshape | (8, 1024, 512) | Heads concatenated back |
| `self.proj(y)` | (8, 1024, 512) | Final output, back to Block |